In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = "../data/raw/complaints.csv"
OUTPUT_FILE = "../data/processed/filtered_complaints.csv"
CHUNK_SIZE = 50000

TARGET_PRODUCTS = [
    "Credit card",
    "Personal loan",
    "Savings account",
    "Money transfer"
]

# ============================================================
# TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove common boilerplate phrases
    boilerplate_patterns = [
        r"i am writing to file a complaint",
        r"i would like to file a complaint",
        r"this complaint is regarding",
        r"i am submitting this complaint"
    ]

    for pattern in boilerplate_patterns:
        text = re.sub(pattern, "", text)

    # Remove special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ============================================================
# EDA VARIABLES
# ============================================================

product_counter = Counter()
with_narrative = 0
without_narrative = 0

word_counts_all = []
processed_chunks = []

# ============================================================
# PROCESS DATA IN CHUNKS
# ============================================================

print("Processing dataset in chunks...")

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):

    print(f"Processing chunk {i+1}")

    # --------------------------------------------------------
    # Product Distribution
    # --------------------------------------------------------
    product_counter.update(
        chunk["Product"].value_counts().to_dict()
    )

    # --------------------------------------------------------
    # Narrative Availability
    # --------------------------------------------------------
    with_narrative += chunk["Consumer complaint narrative"].notna().sum()

    without_narrative += chunk["Consumer complaint narrative"].isna().sum()

    # --------------------------------------------------------
    # Narrative Length Before Filtering
    # --------------------------------------------------------
    temp_lengths = (
        chunk["Consumer complaint narrative"]
        .fillna("")
        .astype(str)
        .apply(lambda x: len(x.split()))
    )

    word_counts_all.extend(temp_lengths.tolist())

    # --------------------------------------------------------
    # Filter Products
    # --------------------------------------------------------
    filtered_chunk = chunk[
        chunk["Product"].isin(TARGET_PRODUCTS)
    ].copy()

    # --------------------------------------------------------
    # Remove Empty Narratives
    # --------------------------------------------------------
    filtered_chunk = filtered_chunk[
        filtered_chunk["Consumer complaint narrative"].notna()
    ]

    filtered_chunk = filtered_chunk[
        filtered_chunk["Consumer complaint narrative"].str.strip() != ""
    ]

    # --------------------------------------------------------
    # Clean Text
    # --------------------------------------------------------
    filtered_chunk["cleaned_narrative"] = (
        filtered_chunk["Consumer complaint narrative"]
        .apply(clean_text)
    )

    # --------------------------------------------------------
    # Word Count After Cleaning
    # --------------------------------------------------------
    filtered_chunk["word_count"] = (
        filtered_chunk["cleaned_narrative"]
        .str.split()
        .str.len()
    )

    processed_chunks.append(filtered_chunk)

# ============================================================
# COMBINE PROCESSED DATA
# ============================================================

filtered_df = pd.concat(
    processed_chunks,
    ignore_index=True
)

# ============================================================
# SAVE CLEANED DATASET
# ============================================================

filtered_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved filtered dataset:")
print(OUTPUT_FILE)

# ============================================================
# EDA RESULTS
# ============================================================

print("\n" + "="*60)
print("EDA SUMMARY")
print("="*60)

print("\nTotal complaints with narrative:")
print(with_narrative)

print("\nTotal complaints without narrative:")
print(without_narrative)

print("\nTop Product Categories:")
print(pd.Series(product_counter).sort_values(ascending=False).head(20))

print("\nNarrative Length Statistics:")
print(pd.Series(word_counts_all).describe())

# ============================================================
# VISUALIZATION 1: PRODUCT DISTRIBUTION
# ============================================================

plt.figure(figsize=(12,6))

pd.Series(product_counter) \
    .sort_values(ascending=False) \
    .head(10) \
    .plot(kind="bar")

plt.title("Top 10 Complaint Products")
plt.ylabel("Number of Complaints")
plt.xlabel("Product")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ============================================================
# VISUALIZATION 2: NARRATIVE LENGTH DISTRIBUTION
# ============================================================

plt.figure(figsize=(12,6))

sns.histplot(
    word_counts_all,
    bins=50,
    kde=True
)

plt.title("Consumer Narrative Length Distribution")
plt.xlabel("Word Count")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

# ============================================================
# FILTERED DATA SUMMARY
# ============================================================

print("\n" + "="*60)
print("FILTERED DATASET SUMMARY")
print("="*60)

print(f"\nShape: {filtered_df.shape}")

print("\nProducts retained:")
print(filtered_df["Product"].value_counts())

print("\nCleaned narrative statistics:")
print(filtered_df["word_count"].describe())

print("\nSample cleaned narratives:")
display(
    filtered_df[
        ["Product", "cleaned_narrative"]
    ].head()
)

Processing dataset in chunks...
Processing chunk 1
Processing chunk 2
Processing chunk 3
Processing chunk 4


C:\Users\Test\AppData\Local\Temp\ipykernel_5976\368506873.py:69: DtypeWarning: Columns (0: Consumer disputed?) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processing chunk 5
Processing chunk 6
Processing chunk 7
Processing chunk 8
Processing chunk 9
Processing chunk 10
Processing chunk 11
Processing chunk 12
Processing chunk 13
Processing chunk 14
Processing chunk 15
Processing chunk 16
Processing chunk 17
Processing chunk 18
Processing chunk 19
Processing chunk 20
Processing chunk 21
Processing chunk 22
Processing chunk 23
Processing chunk 24
Processing chunk 25
Processing chunk 26
Processing chunk 27
Processing chunk 28
Processing chunk 29
Processing chunk 30
Processing chunk 31
Processing chunk 32
Processing chunk 33
Processing chunk 34
Processing chunk 35
Processing chunk 36
Processing chunk 37
Processing chunk 38
Processing chunk 39
Processing chunk 40
Processing chunk 41
Processing chunk 42
Processing chunk 43
Processing chunk 44
Processing chunk 45
Processing chunk 46
Processing chunk 47
Processing chunk 48
Processing chunk 49
Processing chunk 50
Processing chunk 51
Processing chunk 52
Processing chunk 53
Processing chunk 54
Proce

C:\Users\Test\AppData\Local\Temp\ipykernel_5976\368506873.py:69: DtypeWarning: Columns (0: Consumer complaint narrative) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processing chunk 187
Processing chunk 188
Processing chunk 189
Processing chunk 190
Processing chunk 191
Processing chunk 192
Processing chunk 193


OSError: Cannot save file into a non-existent directory: 'data'